In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -U langchain langchain-community langchain-core transformers==4.52.4 fastapi uvicorn pyngrok accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.4/144.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-a

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re

In [4]:
model_path = "microsoft/Phi-4-mini-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

In [5]:
def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [6]:
full_name_schema = ResponseSchema(
    name="full_name",
    description="The Full name of the candidate."
)
email_schema = ResponseSchema(
    name="email",
    description="The Full Email of the candidate."
)
education_schema = ResponseSchema(
    name="education",
    description="A list of three items : { 'degree' : candidate's degree name ,  'institution'  : candidate's institution name ,  'year' : candidate's degree year}."
)
skills_schema = ResponseSchema(
    name="skills",
    description="A list of the candidate's skills."
)
experience_schema = ResponseSchema(
    name="experience",
    description="A list of three items : { 'role'  : candidate's working role name , 'company'  : candidate's working company name , 'years' : candidate's working years}."
)
projects_schema = ResponseSchema(
    name="projects",
    description="A list of the two items : {'name'  : candidate's project name , 'summary': candidate's project summary}."
)


response_schemas = [full_name_schema,
                    email_schema,
                    education_schema,
                    skills_schema,
                    experience_schema,
                    projects_schema]

In [7]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [8]:
cv_extraction_template = """
You are a company's hr smart assistant that extracts information from the candidate's Resumes.

Extract the name of the candidate,the email of the candidate, the education of the candidate, the skills of the candidate, the experience of the candidate and the projects of the candidate.

Respond ONLY in JSON format as follows:
{format_instructions}

Example Input:
"John Smith – john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020–2023)
- Data Scientist at OpenAI (2023–Present)."

Now extract from the following input:
"{cv_input}"
"""

In [9]:
import re


def extract_json_block(text):

    text = text.strip()

    # Try to extract JSON from ```json ... ```
    pattern = r"```(?:json)?\s*(.*?)\s*```"

    matches = re.findall(
        pattern,
        text,
        re.DOTALL | re.IGNORECASE
    )

    if matches:
        return matches[-1].strip()

    # Try to find a raw JSON object
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1:
        return text[start:end + 1].strip()

    raise ValueError(
        "Could not find JSON in model output.\n\n"
        f"Model output:\n{text}"
    )

In [10]:
def cv_analyzer(cv_input):

    prompt = PromptTemplate(
        template=cv_extraction_template,
        input_variables=[
            "cv_input",
            "format_instructions"
        ]
    ).format(
        cv_input=cv_input,
        format_instructions=format_instructions
    )

    # ==========================================
    # GENERATE MODEL RESPONSE
    # ==========================================

    response = generate_text(
        prompt,
        max_length=3000
    )[0]

    # ==========================================
    # DEBUG 1
    # ==========================================

    print("\n")
    print("=" * 80)
    print("RAW MODEL RESPONSE")
    print("=" * 80)

    print(response)

    print("\n")
    print("=" * 80)
    print("RAW MODEL RESPONSE repr()")
    print("=" * 80)

    print(repr(response))

    print("\n")
    print("=" * 80)

    # ==========================================
    # EXTRACT JSON
    # ==========================================

    json_text = extract_json_block(response)

    print("EXTRACTED JSON")
    print("=" * 80)

    print(json_text)

    print("\n")
    print("=" * 80)

    # ==========================================
    # PARSE JSON
    # ==========================================

    output_data = output_parser.parse(json_text)

    print("PARSING SUCCESSFUL")
    print("=" * 80)

    print(type(output_data))
    print(output_data)

    return output_data

In [11]:
NGROK_TOKEN = "3KCPvunMgHlmrcIZUPZ4s3hi0LC_5pUVico6nozVAG9pSXoXb"
API_KEY = "ahmed772005"

In [12]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

@app.post("/generate")
async def gen(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    return {
        "response": cv_analyzer(data.get("cv_input",""))
    }


In [13]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://amplify-ivy-poster.ngrok-free.dev


INFO:     Started server process [1996]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:52417 (Press CTRL+C to quit)
